# IDH Mutation Classification - Soft-Mask Alpha Sweep + ASL Ablation (grid-winner hp)

ResNet-18, contrast-free MRI (**T2 + FLAIR + ASL**), soft mask, all slices (tumor >= 250 voxels).
Hyperparameters are **fixed at the grid winner** (dropout 0.2, lr 3e-5, weight_decay 1e-4); only the
mask strength alpha varies. Everything is selected on **validation**; test is reported once.

1. **Alpha sweep:** train at **alpha = 0.3 / 0.5 / 0.7** (same hp), pick the best alpha on **val AUC**.
   (alpha=0.5 reuses the grid-winner checkpoint so its number matches the headline.)
2. **Training curves** + **GradCAM++** for the best alpha.
3. **ASL ablation** at the best alpha (3-channel, dropped modalities zeroed): 3 singles + 3 pairs,
   plus the full T2+FLAIR+ASL run (the best-alpha model itself).

Patient-level decision = **mean** of per-slice probabilities. Single 60/20/20 split (seed 63).
Reuses the existing all-slices (v250) cache; everything is written to the Elements drive.
Designed to run unattended overnight (~8 trainings).

In [ ]:
import os
os.environ["TORCH_HOME"]   = "/media/omeryasincur/Elements/omeryasin/torch_cache"   # ImageNet pretrained weights -> Elements
os.environ["MPLCONFIGDIR"] = "/media/omeryasincur/Elements/omeryasin/mpl_cache"      # matplotlib cache -> Elements
import re
import json
import random
import time
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from scipy.ndimage import zoom
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix, roc_curve,
)

warnings.filterwarnings("ignore")

In [ ]:
# --- Paths ---
BASE_DIR    = "/media/omeryasincur/Elements/PKG - UCSF-PDGM Version 5"
DATA_DIR    = os.path.join(BASE_DIR, "UCSF-PDGM-v5")
CSV_PATH    = os.path.join(BASE_DIR, "UCSF-PDGM-metadata_v5.csv")
SPLITS_DIR  = "/media/omeryasincur/Elements/omeryasin/idh_splits"
CACHE_DIR   = "/media/omeryasincur/Elements/omeryasin/cache_whole_brain_allslices_v250"   # on Elements drive (shared with no-mask NB)
RESULTS_DIR = "/media/omeryasincur/Elements/omeryasin/pipeline_softmask3ch_sweep_results"  # on Elements
PLOTS_DIR   = os.path.join(RESULTS_DIR, "plots")
for d in [SPLITS_DIR, CACHE_DIR, RESULTS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

# --- Config (matches the P1-all / P2-all all-slices recipe) ---
CFG = {
    "seed":                 63,
    # Data
    "test_fraction":        0.20,        # 99 patients
    "val_fraction":         0.20,        # 99 patients (of total)
    "min_tumor_voxels":     250,         # a slice is kept if tumor area >= this
    "min_slices_fallback":  5,           # if fewer than this qualify, fall back to top-K by area
    "min_tumor_pixels":     50,
    "clip_percentile":      99.5,
    "brain_bbox_margin":    0.02,
    "modalities":           ["T2", "FLAIR", "ASL"],   # 3-channel; each soft-masked at load time  (keys; shown as T2w/FLAIR/ASL in figures & report)
    "mask_alpha":           0.5,         # soft mask: tumor x1, background x mask_alpha (0=hard mask, 1=no mask)
    "expected_patients":    495,
    # Model input
    "img_size":             224,
    # Training
    "batch_size":           16,
    "num_workers":          4,
    "max_epochs":           40,
    "patience":             999,         # early stopping disabled -> train full 40 epochs
    "use_weighted_sampler": True,        # class balance via sampling, not loss weighting
    "lr":                   3e-5,
    "weight_decay":         1e-4,
    "warmup_epochs":        10,          # long, slow warmup (0.001 -> 1.0)
    # Model
    "pretrained":           True,
    "dropout":              0.2,
    # Augmentation
    "aug_hflip":            True,
    "aug_vflip":            True,
    "aug_rotation":         20,
    "aug_scale":            (0.85, 1.15),
    "aug_gamma":            (0.8, 1.2),
    "aug_noise_std":        0.02,
    # Label smoothing
    "label_smoothing":      0.1,
    # Differential learning-rate multipliers (NO layers frozen)
    "lr_mult_early":        0.1,
    "lr_mult_layer4":       1.0,
    "lr_mult_head":         2.0,
    # Cache
    "force_rebuild_cache":  False,
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name()}")

In [ ]:
BASELINE_PATTERN = re.compile(r'^UCSF-PDGM-\d+$')


def normalize_id(s):
    """Canonical 4-digit padded ID."""
    base = s.replace("_nifti", "")
    return f"UCSF-PDGM-{int(base.split('-')[-1]):04d}"


def is_baseline(s):
    """True if ID/folder is a baseline (preoperative) scan, not a follow-up."""
    return "_FU" not in s and bool(BASELINE_PATTERN.match(s.replace("_nifti", "")))


def load_nifti(path):
    return nib.load(path).get_fdata().astype(np.float32)


def get_brain_bbox(brain_mask_3d, margin=0.02):
    projection = brain_mask_3d.max(axis=2) > 0
    if projection.sum() == 0:
        H, W = brain_mask_3d.shape[:2]
        return 0, H - 1, 0, W - 1
    rows = np.any(projection, axis=1)
    cols = np.any(projection, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    H, W = brain_mask_3d.shape[:2]
    pad_r = int((rmax - rmin) * margin)
    pad_c = int((cmax - cmin) * margin)
    return (max(0, rmin - pad_r), min(H - 1, rmax + pad_r),
            max(0, cmin - pad_c), min(W - 1, cmax + pad_c))


def normalize_volume(volume, brain_mask, clip_pct=99.5):
    """clip -> z-score -> min-max [0,1], background = 0."""
    out = np.zeros_like(volume, dtype=np.float32)
    brain_vox = volume[brain_mask > 0]
    if len(brain_vox) == 0 or brain_vox.std() < 1e-8:
        return out
    lo = np.percentile(brain_vox, 100 - clip_pct)
    hi = np.percentile(brain_vox, clip_pct)
    clipped = np.clip(volume, lo, hi)
    brain_clipped = clipped[brain_mask > 0]
    mu, sigma = brain_clipped.mean(), brain_clipped.std()
    if sigma < 1e-8:
        return out
    z = (clipped - mu) / sigma
    z_brain = z[brain_mask > 0]
    zmin, zmax = z_brain.min(), z_brain.max()
    if zmax - zmin < 1e-8:
        return out
    normalized = (z - zmin) / (zmax - zmin)
    normalized[brain_mask == 0] = 0.0
    return normalized.astype(np.float32)


def resize_slice(img, target_size, order=1):
    h, w = img.shape
    if h == target_size and w == target_size:
        return img
    return zoom(img, (target_size / h, target_size / w), order=order)


def select_top_k_slices(seg_volume, k=10, min_pixels=50):
    areas = []
    for i in range(seg_volume.shape[2]):
        area = int((seg_volume[:, :, i] > 0).sum())
        if area >= min_pixels:
            areas.append((i, area))
    areas.sort(key=lambda x: x[1], reverse=True)
    return areas[:k]


def select_all_slices(seg_volume, min_voxels=100, min_slices=5):
    """All axial slices with tumor area >= min_voxels.
    Fallback: if fewer than min_slices qualify, take the top-min_slices by area
    (so every patient keeps at least some slices). Returns (slice_idx, area) tuples."""
    areas_all = [(i, int((seg_volume[:, :, i] > 0).sum())) for i in range(seg_volume.shape[2])]
    qualifying = [(i, a) for i, a in areas_all if a >= min_voxels]
    if len(qualifying) >= min_slices:
        qualifying.sort(key=lambda x: x[1], reverse=True)
        return qualifying
    areas_all.sort(key=lambda x: x[1], reverse=True)
    return areas_all[:min_slices]


# Self-test
assert normalize_id("UCSF-PDGM-004")        == "UCSF-PDGM-0004"
assert normalize_id("UCSF-PDGM-0004_nifti") == "UCSF-PDGM-0004"
assert is_baseline("UCSF-PDGM-004")          is True
assert is_baseline("UCSF-PDGM-0429_FU003d")  is False
print("Helper functions verified")

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Raw CSV rows: {len(df)}")

# IDH value distribution
print("\nIDH value distribution:")
print(df["IDH"].value_counts())

# Drop follow-up scans
mask_baseline = df["ID"].apply(is_baseline)
followup_ids  = df.loc[~mask_baseline, "ID"].tolist()
print(f"\nFollow-up scans dropped: {len(followup_ids)}")

df = df[mask_baseline].copy().reset_index(drop=True)
df["patient_id"] = df["ID"].apply(normalize_id)

# --- LABEL ENCODING: Mut = 1, WT = 0 ---
df["idh_label"] = (df["IDH"] != "wildtype").astype(int)

csv_pids    = set(df["patient_id"])
folder_pids = {
    normalize_id(p) for p in os.listdir(DATA_DIR)
    if p.startswith("UCSF-PDGM-") and is_baseline(p)
}
overlap = csv_pids & folder_pids

print(f"\nCSV baseline patients:    {len(csv_pids)}")
print(f"Folder baseline patients: {len(folder_pids)}")
print(f"Matching:                 {len(overlap)}")
assert len(overlap) == CFG["expected_patients"], f"Expected {CFG['expected_patients']}, got {len(overlap)}"

print("\nClass distribution (Mut=1, WT=0):")
print(f"  Mut (label=1): {(df['idh_label']==1).sum()}")
print(f"  WT  (label=0): {(df['idh_label']==0).sum()}")
print("\nSanity check passed")

In [ ]:
patient_df = df[["patient_id", "idh_label"]].copy().reset_index(drop=True)

# First split: 80/20 train+val vs test
trainval_df, test_df = train_test_split(
    patient_df,
    test_size=CFG["test_fraction"],
    stratify=patient_df["idh_label"],
    random_state=CFG["seed"],
)

# Second split: from train+val, take val (99/396 ~ 0.25 of the remaining)
val_size_of_remaining = CFG["val_fraction"] / (1.0 - CFG["test_fraction"])
train_df_p, val_df_p = train_test_split(
    trainval_df,
    test_size=val_size_of_remaining,
    stratify=trainval_df["idh_label"],
    random_state=CFG["seed"],
)

train_pids = train_df_p["patient_id"].tolist()
val_pids   = val_df_p["patient_id"].tolist()
test_pids  = test_df["patient_id"].tolist()

# Verify no overlap
assert len(set(train_pids) & set(val_pids)) == 0
assert len(set(train_pids) & set(test_pids)) == 0
assert len(set(val_pids) & set(test_pids)) == 0
assert len(train_pids) + len(val_pids) + len(test_pids) == CFG["expected_patients"]

split_info = {"seed": CFG["seed"], "train": sorted(train_pids),
              "val": sorted(val_pids), "test": sorted(test_pids)}
with open(os.path.join(SPLITS_DIR, "split.json"), "w") as f:
    json.dump(split_info, f, indent=2)


def class_dist(pids):
    sub = patient_df[patient_df["patient_id"].isin(pids)]
    return f"Mut={int((sub['idh_label']==1).sum())}, WT={int((sub['idh_label']==0).sum())}"


print(f"Train: {len(train_pids)} patients ({class_dist(train_pids)})")
print(f"Val:   {len(val_pids)} patients ({class_dist(val_pids)})")
print(f"Test:  {len(test_pids)} patients ({class_dist(test_pids)})")
print(f"\nSplit saved to: {os.path.join(SPLITS_DIR, 'split.json')}")

In [ ]:
manifest_path = os.path.join(CACHE_DIR, "manifest.csv")
label_map = dict(zip(df["patient_id"], df["idh_label"]))

if os.path.exists(manifest_path) and not CFG["force_rebuild_cache"]:
    print(f"Cache exists at {CACHE_DIR}, skipping build.")
else:
    print(f"Building cache at {CACHE_DIR} ...")
    manifest_rows = []
    skipped = []

    for pid_folder in tqdm(sorted(os.listdir(DATA_DIR))):
        if not pid_folder.startswith("UCSF-PDGM-"):
            continue
        base = pid_folder.replace("_nifti", "")
        if "_FU" in base:
            skipped.append((pid_folder, "follow_up_scan"))
            continue
        pid = normalize_id(base)
        if pid not in label_map:
            skipped.append((pid, "no_label"))
            continue

        folder = os.path.join(DATA_DIR, pid_folder)
        try:
            T2_raw    = load_nifti(os.path.join(folder, f"{base}_T2_bias.nii.gz"))
            FLAIR_raw = load_nifti(os.path.join(folder, f"{base}_FLAIR_bias.nii.gz"))
            ASL_raw   = load_nifti(os.path.join(folder, f"{base}_ASL.nii.gz"))
            tumor_seg = load_nifti(os.path.join(folder, f"{base}_tumor_segmentation.nii.gz"))
            brain_seg = load_nifti(os.path.join(folder, f"{base}_brain_segmentation.nii.gz"))
        except Exception as e:
            skipped.append((pid, f"load_error: {e}"))
            continue

        brain_mask = (brain_seg > 0).astype(np.uint8)
        tumor_mask = (tumor_seg > 0).astype(np.uint8)
        rmin, rmax, cmin, cmax = get_brain_bbox(brain_mask, margin=CFG["brain_bbox_margin"])

        T2_n    = normalize_volume(T2_raw,    brain_mask, clip_pct=CFG["clip_percentile"])
        FLAIR_n = normalize_volume(FLAIR_raw, brain_mask, clip_pct=CFG["clip_percentile"])
        ASL_n   = normalize_volume(ASL_raw,   brain_mask, clip_pct=CFG["clip_percentile"])

        top_slices = select_all_slices(tumor_seg, min_voxels=CFG["min_tumor_voxels"],
                                       min_slices=CFG["min_slices_fallback"])
        if not top_slices:
            skipped.append((pid, "no_tumor_slice"))
            continue

        patient_cache_dir = os.path.join(CACHE_DIR, pid)
        os.makedirs(patient_cache_dir, exist_ok=True)

        for rank, (k, area) in enumerate(top_slices):
            slice_path = os.path.join(patient_cache_dir, f"slice_{rank:02d}_k{k:03d}.npz")
            np.savez_compressed(
                slice_path,
                T2    = T2_n[rmin:rmax+1, cmin:cmax+1, k].astype(np.float16),
                FLAIR = FLAIR_n[rmin:rmax+1, cmin:cmax+1, k].astype(np.float16),
                ASL   = ASL_n[rmin:rmax+1, cmin:cmax+1, k].astype(np.float16),
                mask  = tumor_mask[rmin:rmax+1, cmin:cmax+1, k].astype(np.uint8),
                brain = brain_mask[rmin:rmax+1, cmin:cmax+1, k].astype(np.uint8),
            )
            manifest_rows.append({
                "patient_id": pid,
                "slice_path": slice_path,
                "k_axial":    int(k),
                "rank":       int(rank),
                "tumor_area": int(area),
                "idh_label":  int(label_map[pid]),
            })

    pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)
    print(f"\nCache build complete: {len(manifest_rows)} slices")

manifest = pd.read_csv(manifest_path)
manifest["idh_label"] = manifest["patient_id"].map(label_map)
manifest.to_csv(manifest_path, index=False)
print(f"\nLoaded manifest: {len(manifest)} slices, {manifest['patient_id'].nunique()} patients")
print(f"Slice-level: Mut={int((manifest['idh_label']==1).sum())}, "
      f"WT={int((manifest['idh_label']==0).sum())}")

In [ ]:
# --- Preview: full modalities vs. masked input (what the model will see) ---
rng = np.random.default_rng(CFG["seed"])
mut_pid = rng.choice(manifest[manifest["idh_label"] == 1]["patient_id"].unique())

rows = manifest[manifest["patient_id"] == mut_pid]
row  = rows.iloc[len(rows) // 2]
data = np.load(row["slice_path"])
m = data["mask"].astype(np.float32)
alpha = CFG["mask_alpha"]
soft = alpha + (1.0 - alpha) * m              # tumor -> 1.0, background -> alpha
ch = {"T2": data["T2"], "FLAIR": data["FLAIR"], "ASL": data["ASL"]}

fig, axes = plt.subplots(2, 4, figsize=(18, 9)); fig.patch.set_facecolor("white")
for ax, name in zip(axes[0, :3], ["T2", "FLAIR", "ASL"]):
    ax.imshow(np.rot90(ch[name]), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{name} (full)", fontsize=13, fontweight="bold"); ax.axis("off")
axes[0, 3].imshow(np.rot90(m), cmap="gray", vmin=0, vmax=1)
axes[0, 3].set_title("Tumor mask", fontsize=13, fontweight="bold"); axes[0, 3].axis("off")
for ax, name in zip(axes[1, :3], ["T2", "FLAIR", "ASL"]):
    ax.imshow(np.rot90(ch[name] * soft), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{name} x soft", fontsize=13, fontweight="bold"); ax.axis("off")
rgb = np.stack([np.rot90(ch["T2"] * soft), np.rot90(ch["FLAIR"] * soft), np.rot90(ch["ASL"] * soft)], axis=-1)
axes[1, 3].imshow(np.clip(rgb, 0, 1))
axes[1, 3].set_title("soft-masked T2/FLAIR/ASL (RGB)", fontsize=13, fontweight="bold"); axes[1, 3].axis("off")
fig.suptitle(f"{mut_pid}  |  k={row['k_axial']}  |  tumor area={row['tumor_area']}  |  "
             f"background weight (alpha) = {alpha}", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
class IDHSliceDataset(Dataset):
    """3-channel dataset with a SOFT tumor mask: the tumor region is kept at full
    intensity (x1) and everything outside is attenuated (x mask_alpha, e.g. 0.3)
    instead of being zeroed. This keeps whole-brain context while emphasizing the
    tumor. There is NO separate mask channel -> input stays 3-channel."""

    def __init__(self, manifest_df, cfg, augment=False):
        self.df       = manifest_df.reset_index(drop=True)
        self.cfg      = cfg
        self.augment  = augment
        self.img_size = cfg["img_size"]
        self.mods     = cfg["modalities"]            # ["T2", "FLAIR", "ASL"]

    def __len__(self):
        return len(self.df)

    def _augment(self, img):
        if self.cfg["aug_hflip"] and random.random() < 0.5:
            img = TF.hflip(img)
        if self.cfg.get("aug_vflip", False) and random.random() < 0.5:
            img = TF.vflip(img)
        if self.cfg["aug_rotation"] > 0:
            angle = random.uniform(-self.cfg["aug_rotation"], self.cfg["aug_rotation"])
            img = TF.rotate(img, angle, interpolation=TF.InterpolationMode.BILINEAR)
        if self.cfg["aug_scale"] is not None:
            sc = random.uniform(*self.cfg["aug_scale"])
            img = TF.affine(img, angle=0, translate=[0, 0], scale=sc, shear=0,
                            interpolation=TF.InterpolationMode.BILINEAR)
        if random.random() < 0.5:
            gamma = random.uniform(*self.cfg["aug_gamma"])
            img = torch.clamp(img, 0, 1) ** gamma
        if random.random() < 0.3 and self.cfg["aug_noise_std"] > 0:
            img = img + torch.randn_like(img) * self.cfg["aug_noise_std"]
            img = torch.clamp(img, 0, 1)
        return img

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        data = np.load(row["slice_path"])
        tumor = data["mask"].astype(np.float32)       # binary tumor mask (1 = tumor, 0 = rest)
        alpha = self.cfg["mask_alpha"]
        # Soft weight map: tumor -> 1.0, background -> alpha (context dimmed, not removed).
        soft = alpha + (1.0 - alpha) * tumor
        modality_resized = [
            resize_slice(data[m].astype(np.float32) * soft, self.img_size, order=1)
            for m in self.mods
        ]
        img = np.stack(modality_resized, axis=0).astype(np.float32)
        img_tensor = torch.from_numpy(img)
        if self.augment:
            img_tensor = self._augment(img_tensor)
        return {
            "image":      img_tensor,
            "label":      torch.tensor(row["idh_label"], dtype=torch.float32),
            "patient_id": row["patient_id"],
        }


# Sanity check
_test_ds = IDHSliceDataset(manifest, CFG, augment=True)
_s = _test_ds[0]
print(f"Dataset size: {len(_test_ds)}")
print(f"Sample 0: image shape={tuple(_s['image'].shape)}, "
      f"range=[{_s['image'].min():.3f}, {_s['image'].max():.3f}], "
      f"label={_s['label'].item()}, pid={_s['patient_id']}")
assert _s["image"].shape == (3, CFG["img_size"], CFG["img_size"])

In [ ]:
def build_model(pretrained=True, dropout=0.5):
    weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = torchvision.models.resnet18(weights=weights)
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(512, 1),
    )
    return model


_m = build_model(pretrained=CFG["pretrained"], dropout=CFG["dropout"]).to(DEVICE)
total = sum(p.numel() for p in _m.parameters())
print(f"Model: ResNet-18 (3-channel input, binary head, dropout={CFG['dropout']})")
print(f"  Parameters: {total:,}")
_m.eval()
with torch.no_grad():
    _x = torch.randn(2, 3, CFG["img_size"], CFG["img_size"]).to(DEVICE)
    _y = _m(_x)
print(f"  Forward: {tuple(_x.shape)} -> {tuple(_y.shape)}")
assert _y.shape == (2, 1)
del _m

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, label_smoothing=0.0):
    model.train()
    total_loss, n = 0.0, 0
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        if label_smoothing > 0:
            labels = labels * (1.0 - 2.0 * label_smoothing) + label_smoothing
        optimizer.zero_grad(set_to_none=True)
        logits = model(images).squeeze(-1)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        bs = labels.size(0)
        total_loss += loss.item() * bs
        n += bs
    return total_loss / n


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, n = 0.0, 0
    all_logits, all_labels, all_pids = [], [], []
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        logits = model(images).squeeze(-1)
        loss = criterion(logits, labels)
        bs = labels.size(0)
        total_loss += loss.item() * bs
        n += bs
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())
        all_pids.extend(batch["patient_id"])
    return {
        "loss":   total_loss / n,
        "logits": torch.cat(all_logits),
        "labels": torch.cat(all_labels),
        "pids":   all_pids,
    }


print("Train/validate functions defined")

In [ ]:
def aggregate_to_patient(pids, slice_probs, slice_labels):
    """Mean probability across slices for each patient."""
    data = defaultdict(lambda: {"probs": [], "label": None})
    for pid, p, l in zip(pids, slice_probs, slice_labels):
        data[pid]["probs"].append(float(p))
        data[pid]["label"] = int(l)
    pids_out, probs_out, labels_out = [], [], []
    for pid, d in data.items():
        pids_out.append(pid)
        probs_out.append(np.mean(d["probs"]))
        labels_out.append(d["label"])
    return np.array(pids_out), np.array(probs_out), np.array(labels_out)


def find_f1_threshold(labels, probs):
    """Threshold that maximizes F1 (balanced precision/recall). Selected on val."""
    if len(np.unique(labels)) < 2:
        return 0.5
    best_f1, best_thr = -1.0, 0.5
    candidate_thrs = sorted(set(np.round(probs, 4).tolist() + [0.5]))
    for thr in candidate_thrs:
        preds = (probs >= thr).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return float(best_thr)


def find_youden_threshold(labels, probs):
    """Kept for reference; F1 is primary."""
    if len(np.unique(labels)) < 2:
        return 0.5
    fpr, tpr, thresholds = roc_curve(labels, probs)
    return float(thresholds[np.argmax(tpr - fpr)])


def compute_metrics(labels, probs, threshold):
    """Mut=1 convention: Recall = sensitivity for Mut, Specificity = correct WT."""
    labels = np.asarray(labels).astype(int)
    preds  = (probs >= threshold).astype(int)
    try:
        auc = roc_auc_score(labels, probs) if len(np.unique(labels)) >= 2 else float("nan")
    except ValueError:
        auc = float("nan")
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        "AUC":         auc,
        "Accuracy":    accuracy_score(labels, preds),
        "Precision":   precision_score(labels, preds, zero_division=0),
        "Recall":      recall_score(labels, preds, zero_division=0),
        "F1":          f1_score(labels, preds, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "Threshold":   threshold,
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
    }


print("Metrics + thresholds defined (F1-optimal = primary)")

In [ ]:
# 3-channel SOFT-MASKED input; channels not in `active` are ZEROED.
# Used for BOTH the alpha sweep (active = all three) and the ASL ablation (subsets).
class AblationZeroDataset(Dataset):
    """Channel order is always [T2, FLAIR, ASL]; e.g. active={'FLAIR'} -> [0, FLAIR*soft, 0].
    Soft mask: tumor -> x1, background -> x alpha. Dropped channels are exactly 0
    (re-zeroed after augmentation so noise can't leak in). No separate mask channel."""
    ALL = ["T2", "FLAIR", "ASL"]
    def __init__(self, manifest_df, cfg, active, augment=False):
        self.df = manifest_df.reset_index(drop=True)
        self.cfg = cfg
        self.augment = augment
        self.img_size = cfg["img_size"]
        self.active = list(active)
        self.alpha = cfg.get("mask_alpha", 0.5)
    def __len__(self): return len(self.df)
    def _augment(self, img):
        if self.cfg["aug_hflip"] and random.random() < 0.5: img = TF.hflip(img)
        if self.cfg.get("aug_vflip", False) and random.random() < 0.5: img = TF.vflip(img)
        if self.cfg["aug_rotation"] > 0:
            angle = random.uniform(-self.cfg["aug_rotation"], self.cfg["aug_rotation"])
            img = TF.rotate(img, angle, interpolation=TF.InterpolationMode.BILINEAR)
        if self.cfg["aug_scale"] is not None:
            sc = random.uniform(*self.cfg["aug_scale"])
            img = TF.affine(img, angle=0, translate=[0, 0], scale=sc, shear=0,
                            interpolation=TF.InterpolationMode.BILINEAR)
        if random.random() < 0.5:
            gamma = random.uniform(*self.cfg["aug_gamma"]); img = torch.clamp(img, 0, 1) ** gamma
        if random.random() < 0.3 and self.cfg["aug_noise_std"] > 0:
            img = img + torch.randn_like(img) * self.cfg["aug_noise_std"]; img = torch.clamp(img, 0, 1)
        return img
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; data = np.load(row["slice_path"])
        tumor = data["mask"].astype(np.float32)
        soft = self.alpha + (1.0 - self.alpha) * tumor
        chans = []
        for m in self.ALL:                                       # fixed channel order
            if m in self.active:
                chans.append(resize_slice(data[m].astype(np.float32) * soft, self.img_size, order=1))
            else:
                chans.append(np.zeros((self.img_size, self.img_size), dtype=np.float32))   # zeroed channel
        img = torch.from_numpy(np.stack(chans, axis=0).astype(np.float32))
        if self.augment:
            img = self._augment(img)
            for ci, m in enumerate(self.ALL):                    # re-zero dropped channels (remove aug noise)
                if m not in self.active:
                    img[ci].zero_()
        return {"image": img, "label": torch.tensor(row["idh_label"], dtype=torch.float32),
                "patient_id": row["patient_id"]}

print("Soft + zeroing dataset ready (3-channel; alpha via cfg, dropped channels = 0)")

In [ ]:
def train_scenario(name, active, alpha, train_mf, val_mf, cfg, device, base_dir):
    """Train one 3-channel soft-mask model and record history. active=all -> sweep model;
    active=subset -> ablation. Best epoch selected by patient-level val AUC."""
    sdir = os.path.join(base_dir, name); os.makedirs(sdir, exist_ok=True)
    rcfg = dict(cfg); rcfg["mask_alpha"] = alpha
    print(f"\n{'='*66}\n  {name}  (active={active}, alpha={alpha})\n{'='*66}")
    train_ds = AblationZeroDataset(train_mf, rcfg, active, augment=True)
    val_ds   = AblationZeroDataset(val_mf,   rcfg, active, augment=False)
    labels = train_mf["idh_label"].values
    n_mut = (labels == 1).sum(); n_wt = (labels == 0).sum()
    w = np.where(labels == 1, 1.0 / n_mut, 1.0 / n_wt)
    sampler = torch.utils.data.WeightedRandomSampler(w, num_samples=len(labels), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], sampler=sampler,
                              num_workers=cfg["num_workers"], pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False,
                            num_workers=cfg["num_workers"], pin_memory=True)
    model = build_model(pretrained=cfg["pretrained"], dropout=cfg["dropout"]).to(device)   # 3-channel, same arch
    early = (list(model.conv1.parameters()) + list(model.bn1.parameters()) +
             list(model.layer1.parameters()) + list(model.layer2.parameters()) + list(model.layer3.parameters()))
    optimizer = torch.optim.AdamW([
        {"params": early,                     "lr": cfg["lr"] * cfg["lr_mult_early"]},
        {"params": model.layer4.parameters(), "lr": cfg["lr"] * cfg["lr_mult_layer4"]},
        {"params": model.fc.parameters(),     "lr": cfg["lr"] * cfg["lr_mult_head"]},
    ], weight_decay=cfg["weight_decay"])
    criterion = nn.BCEWithLogitsLoss()
    we = cfg["warmup_epochs"]
    def lr_lambda(epoch):
        if epoch < we: return 0.001 + (1.0 - 0.001) * (epoch / we)
        prog = (epoch - we) / max(1, (cfg["max_epochs"] - we)); return 0.5 * (1 + np.cos(np.pi * prog))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    history = {"epoch": [], "train_loss": [], "val_loss": [], "val_auc": []}
    best_auc = -1.0; best_ckpt = os.path.join(sdir, "best_model.pt")
    for epoch in range(1, cfg["max_epochs"] + 1):
        tl = train_one_epoch(model, train_loader, criterion, optimizer, device, label_smoothing=cfg["label_smoothing"])
        scheduler.step()
        vo = validate(model, val_loader, criterion, device)
        _, vp, vy = aggregate_to_patient(vo["pids"], torch.sigmoid(vo["logits"]).numpy(), vo["labels"].numpy().astype(int))
        va = roc_auc_score(vy, vp) if len(np.unique(vy)) >= 2 else 0.5
        history["epoch"].append(epoch); history["train_loss"].append(tl)
        history["val_loss"].append(vo["loss"]); history["val_auc"].append(va)
        if va > best_auc + 1e-4:
            best_auc = va; torch.save(model.state_dict(), best_ckpt)
    json.dump(history, open(os.path.join(sdir, "history.json"), "w"), indent=2)
    print(f"  best val_AUC (patient) = {best_auc:.4f}")
    return {"name": name, "active": active, "alpha": alpha, "best_ckpt": best_ckpt,
            "val_loader": val_loader, "model": model, "history": history, "rcfg": rcfg, "sdir": sdir}


def eval_scenario(res, test_mf, cfg, device):
    """Evaluate a trained scenario on val + test; also return slice-level probs for aggregation."""
    model = res["model"]; model.load_state_dict(torch.load(res["best_ckpt"], map_location=device)); model.eval()
    test_ds = AblationZeroDataset(test_mf, res["rcfg"], res["active"], augment=False)
    test_loader = DataLoader(test_ds, batch_size=cfg["batch_size"], shuffle=False,
                             num_workers=cfg["num_workers"], pin_memory=True)
    criterion = nn.BCEWithLogitsLoss()
    vo = validate(model, res["val_loader"], criterion, device)
    to = validate(model, test_loader, criterion, device)
    v_sp = torch.sigmoid(vo["logits"]).numpy(); v_sl = vo["labels"].numpy().astype(int)
    t_sp = torch.sigmoid(to["logits"]).numpy(); t_sl = to["labels"].numpy().astype(int)
    _, vp, vy = aggregate_to_patient(vo["pids"], v_sp, v_sl)
    _, tp, ty = aggregate_to_patient(to["pids"], t_sp, t_sl)
    thr = find_f1_threshold(vy, vp)
    vm = compute_metrics(vy, vp, thr); tm = compute_metrics(ty, tp, thr)
    print(f"  thr={thr:.3f} | test AUC={tm['AUC']:.3f} F1={tm['F1']:.3f} P={tm['Precision']:.3f} "
          f"R={tm['Recall']:.3f} Spec={tm['Specificity']:.3f} Acc={tm['Accuracy']:.3f} "
          f"(TP={tm['TP']} TN={tm['TN']} FP={tm['FP']} FN={tm['FN']})")
    return {"f1_threshold": thr, "val": vm, "test": tm,
            "val_probs": (list(vo["pids"]), v_sp, v_sl),
            "test_probs": (list(to["pids"]), t_sp, t_sl)}

print("train_scenario + eval_scenario ready")

In [ ]:
train_mf = manifest[manifest["patient_id"].isin(train_pids)].copy().reset_index(drop=True)
val_mf   = manifest[manifest["patient_id"].isin(val_pids)].copy().reset_index(drop=True)
test_mf  = manifest[manifest["patient_id"].isin(test_pids)].copy().reset_index(drop=True)

# Hyperparameters fixed at the grid winner (found at alpha=0.5); only alpha varies here.
best_train_cfg = dict(CFG)
best_train_cfg["dropout"]      = 0.2
best_train_cfg["lr"]           = 3e-5
best_train_cfg["weight_decay"] = 1e-4

SWEEP_DIR = "/media/omeryasincur/Elements/omeryasin/pipeline_softmask3ch_alphasweep"
os.makedirs(SWEEP_DIR, exist_ok=True)
ALPHAS = [0.3, 0.5, 0.7]

# Reuse the grid-winner checkpoint for alpha=0.5 so its number stays identical to the headline.
GRID_BEST_CKPT = "/media/omeryasincur/Elements/omeryasin/pipeline_softmask3ch_grid/dp0.2_lr3.0e-05_wd1.0e-04/best_model.pt"
GRID_BEST_HIST = "/media/omeryasincur/Elements/omeryasin/pipeline_softmask3ch_grid/dp0.2_lr3.0e-05_wd1.0e-04/history.json"

def _reuse_alpha05():
    rcfg = dict(best_train_cfg); rcfg["mask_alpha"] = 0.5
    sdir = os.path.join(SWEEP_DIR, "alpha0.5"); os.makedirs(sdir, exist_ok=True)
    vl = DataLoader(AblationZeroDataset(val_mf, rcfg, ["T2", "FLAIR", "ASL"], augment=False),
                    batch_size=best_train_cfg["batch_size"], shuffle=False,
                    num_workers=best_train_cfg["num_workers"], pin_memory=True)
    m = build_model(pretrained=CFG["pretrained"], dropout=best_train_cfg["dropout"]).to(DEVICE)
    m.load_state_dict(torch.load(GRID_BEST_CKPT, map_location=DEVICE))
    h = json.load(open(GRID_BEST_HIST)) if os.path.exists(GRID_BEST_HIST) else None
    return {"name": "alpha0.5", "active": ["T2", "FLAIR", "ASL"], "alpha": 0.5, "best_ckpt": GRID_BEST_CKPT,
            "val_loader": vl, "model": m, "history": h, "rcfg": rcfg, "sdir": sdir}

sweep = {}
set_seed(CFG["seed"])
for a in ALPHAS:
    if a == 0.5 and os.path.exists(GRID_BEST_CKPT):
        print(f"\n[alpha=0.5] reusing grid-winner checkpoint (keeps headline consistent)")
        r = _reuse_alpha05()
    else:
        r = train_scenario(f"alpha{a}", ["T2", "FLAIR", "ASL"], a, train_mf, val_mf, best_train_cfg, DEVICE, SWEEP_DIR)
    e = eval_scenario(r, test_mf, best_train_cfg, DEVICE)
    sweep[a] = {"res": r, "eval": e}
    del r["model"]; torch.cuda.empty_cache()

rows = []
for a in ALPHAS:
    v = sweep[a]["eval"]["val"]; t = sweep[a]["eval"]["test"]
    rows.append({"alpha": a, "val_AUC": v["AUC"], "val_F1": v["F1"], "test_AUC": t["AUC"],
                 "test_F1": t["F1"], "test_P": t["Precision"], "test_R": t["Recall"],
                 "test_Spec": t["Specificity"], "test_Acc": t["Accuracy"]})
sweep_df = pd.DataFrame(rows)
best_alpha = float(sweep_df.sort_values(["val_AUC", "val_F1"], ascending=False).iloc[0]["alpha"])
sweep_df.to_csv(os.path.join(SWEEP_DIR, "alpha_sweep_results.csv"), index=False)

print(f"\n{'='*94}\n  ALPHA SWEEP (fixed grid-winner hp: dp0.2 / lr3e-5 / wd1e-4, all slices) - VAL-selected\n{'='*94}")
print(f"{'alpha':>6}{'val_AUC':>9}{'val_F1':>8} | {'test_AUC':>10}{'test_F1':>9}{'test_P':>8}{'test_R':>8}{'test_Spec':>10}{'test_Acc':>9}")
print("-" * 94)
for _, rr in sweep_df.iterrows():
    star = "  <- best (val)" if abs(rr["alpha"] - best_alpha) < 1e-9 else ""
    print(f"{rr['alpha']:>6.1f}{rr['val_AUC']:>9.3f}{rr['val_F1']:>8.3f} | {rr['test_AUC']:>10.3f}{rr['test_F1']:>9.3f}{rr['test_P']:>8.3f}{rr['test_R']:>8.3f}{rr['test_Spec']:>10.3f}{rr['test_Acc']:>9.3f}{star}")
print(f"\nBest alpha (selected on validation): {best_alpha}")

In [ ]:
hist = sweep[best_alpha]["res"]["history"]
if hist is None:
    print(f"(no saved history for alpha={best_alpha}; skipping curves)")
else:
    ep = hist["epoch"]; best_ep = int(np.argmax(hist["val_auc"])) + 1
    fig, ax1 = plt.subplots(figsize=(8.5, 4.8)); fig.patch.set_facecolor("white")
    ax1.plot(ep, hist["train_loss"], color="#B26A00", lw=2, label="train loss")
    ax1.plot(ep, hist["val_loss"],   color="#1C7293", lw=2, label="val loss")
    ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.grid(alpha=0.3)
    ax2 = ax1.twinx()
    ax2.plot(ep, hist["val_auc"], color="#2E8B57", lw=2, ls="--", label="val AUC (patient)")
    ax2.set_ylabel("val AUC (patient-level)")
    ax2.axvline(best_ep, color="gray", ls=":", lw=1.5)
    ax2.annotate(f"best epoch {best_ep} (val AUC {max(hist['val_auc']):.3f})",
                 xy=(best_ep, max(hist["val_auc"])), xytext=(0.40, 0.16), textcoords="axes fraction",
                 fontsize=9, arrowprops=dict(arrowstyle="->", color="gray"))
    h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="center right", fontsize=9)
    plt.title(f"Best alpha = {best_alpha} (grid-winner hp) - training curves", fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.savefig(os.path.join(SWEEP_DIR, f"alpha{best_alpha}_curves.png"), dpi=140, bbox_inches="tight"); plt.show()
    print(f"val AUC peak = {max(hist['val_auc']):.3f} at epoch {best_ep}")

In [ ]:
import torch.nn.functional as F

class GradCAMpp:
    """target_class=1 -> regions supporting MUT; target_class=0 -> regions supporting WT."""
    def __init__(self, model, target_layer):
        self.model = model.eval(); self.target_layer = target_layer
        self.activations = None; self.gradients = None
        target_layer.register_forward_hook(self._fwd_hook)
        target_layer.register_full_backward_hook(self._bwd_hook)
    def _fwd_hook(self, m, i, o): self.activations = o.detach()
    def _bwd_hook(self, m, gi, go): self.gradients = go[0].detach()
    def __call__(self, x, target_class=1):
        x = x.requires_grad_(True)
        logits = self.model(x).squeeze(-1)
        self.model.zero_grad()
        sign = 1.0 if target_class == 1 else -1.0
        (sign * logits.sum()).backward()
        grads = self.gradients; acts = self.activations
        g2 = grads ** 2; g3 = grads ** 3
        denom = (acts * g3).sum(dim=[2, 3], keepdim=True)
        alpha = g2 / (2.0 * g2 + denom + 1e-8)
        weights = (alpha * F.relu(grads)).sum(dim=[2, 3], keepdim=True)
        cam = F.relu((weights * acts).sum(dim=1))
        cam = F.interpolate(cam.unsqueeze(1), size=x.shape[-2:], mode="bilinear", align_corners=False).squeeze(1)
        cmin = cam.amin(dim=[1, 2], keepdim=True); cmax = cam.amax(dim=[1, 2], keepdim=True)
        cam = (cam - cmin) / (cmax - cmin + 1e-8)
        return cam.detach().cpu().numpy(), torch.sigmoid(logits).detach().cpu().numpy()


def gradcam_grid_soft(model, manifest_df, target_layer, device, thr, alpha, n_per_class=8, ncols=4, cfg=CFG, seed=42):
    cam = GradCAMpp(model, target_layer); S = cfg["img_size"]
    r0 = manifest_df[manifest_df["rank"] == 0]
    idx1 = r0[r0["idh_label"] == 1].sample(n=min(n_per_class, int((r0["idh_label"] == 1).sum())), random_state=seed).index.tolist()
    idx0 = r0[r0["idh_label"] == 0].sample(n=min(n_per_class, int((r0["idh_label"] == 0).sum())), random_state=seed).index.tolist()
    indices = idx1 + idx0
    nrows = int(np.ceil(len(indices) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.1, nrows * 3.4)); fig.patch.set_facecolor("white")
    axes = np.asarray(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for ax, idx in zip(axes, indices):
        row = manifest_df.iloc[idx]; data = np.load(row["slice_path"])
        pid = row["patient_id"]; label = int(row["idh_label"])
        tumor = data["mask"].astype(np.float32); soft = alpha + (1.0 - alpha) * tumor
        in_ch = [resize_slice(data[m].astype(np.float32) * soft, S, order=1) for m in ["T2", "FLAIR", "ASL"]]
        full  = resize_slice(data["T2"].astype(np.float32), S, order=1)
        mask  = resize_slice(tumor, S, order=0); brain = resize_slice(data["brain"].astype(np.float32), S, order=0)
        x = torch.from_numpy(np.stack(in_ch, axis=0).astype(np.float32)).unsqueeze(0).to(device)
        hm, prob = cam(x, target_class=label); hm = hm[0] * brain; p = float(prob[0]); pred = int(p >= thr)
        ax.imshow(np.rot90(full), cmap="gray", vmin=0, vmax=1)
        ax.imshow(np.rot90(hm), cmap="jet", alpha=0.5)
        ax.contour(np.rot90(mask), levels=[0.5], colors="white", linewidths=1.4)
        ok = "\u2713" if pred == label else "\u2717"
        tcol = "#1C7293" if label == 1 else "#B26A00"
        ax.set_title(f"{pid}\nTrue {'Mut' if label==1 else 'WT'} | Pred {'Mut' if pred==1 else 'WT'} ({p:.2f}) [{ok}]",
                     fontsize=9, fontweight="bold", color=tcol)
    fig.suptitle(f"GradCAM++ (best alpha = {alpha}, soft-mask) - heatmap over T2 + tumor contour   |   top = Mut, bottom = WT",
                 fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.savefig(os.path.join(SWEEP_DIR, f"gradcam_alpha{alpha}.png"), dpi=140, bbox_inches="tight"); plt.show()


gmodel = build_model(pretrained=CFG["pretrained"], dropout=best_train_cfg["dropout"]).to(DEVICE)
gmodel.load_state_dict(torch.load(sweep[best_alpha]["res"]["best_ckpt"], map_location=DEVICE)); gmodel.eval()
gradcam_grid_soft(gmodel, test_mf, gmodel.layer4, DEVICE, sweep[best_alpha]["eval"]["f1_threshold"], alpha=best_alpha, n_per_class=8, ncols=4)

In [ ]:
ABLATION_DIR = "/media/omeryasincur/Elements/omeryasin/pipeline_softmask3ch_alphasweep_ablation"
os.makedirs(ABLATION_DIR, exist_ok=True)
SCENARIOS = [("T2", ["T2"]), ("FLAIR", ["FLAIR"]), ("ASL", ["ASL"]),
             ("T2+FLAIR", ["T2", "FLAIR"]), ("T2+ASL", ["T2", "ASL"]), ("FLAIR+ASL", ["FLAIR", "ASL"])]

abl_rows = []
set_seed(CFG["seed"])
for name, active in SCENARIOS:
    r = train_scenario(name, active, best_alpha, train_mf, val_mf, best_train_cfg, DEVICE, ABLATION_DIR)
    e = eval_scenario(r, test_mf, best_train_cfg, DEVICE); tm = e["test"]
    abl_rows.append((name, len(active), e["f1_threshold"], tm["AUC"], tm["F1"], tm["Precision"],
                     tm["Recall"], tm["Specificity"], tm["Accuracy"], tm["FP"] + tm["FN"]))
    del r["model"]; torch.cuda.empty_cache()

# full T2+FLAIR+ASL row = the best-alpha model (reuse from the sweep, no retraining)
ftm = sweep[best_alpha]["eval"]["test"]; fthr = sweep[best_alpha]["eval"]["f1_threshold"]
abl_rows.append(("T2+FLAIR+ASL", 3, fthr, ftm["AUC"], ftm["F1"], ftm["Precision"],
                 ftm["Recall"], ftm["Specificity"], ftm["Accuracy"], ftm["FP"] + ftm["FN"]))

print(f"\n{'='*96}\n  ASL ABLATION (best alpha = {best_alpha}, grid-winner hp, ZEROING) - patient-level TEST\n{'='*96}")
print(f"{'scenario':<14}{'act':>4}{'thr':>7}{'AUC':>8}{'F1':>8}{'Prec':>7}{'Rec':>7}{'Spec':>7}{'Acc':>7}{'err':>5}")
print("-" * 96)
for name, ch, thr, auc, f1, p, rec, sp, acc, err in abl_rows:
    star = "  <- full (best-alpha model)" if name == "T2+FLAIR+ASL" else ""
    print(f"{name:<14}{ch:>4}{thr:>7.3f}{auc:>8.3f}{f1:>8.3f}{p:>7.3f}{rec:>7.3f}{sp:>7.3f}{acc:>7.3f}{err:>5d}{star}")

abl_df = pd.DataFrame(abl_rows, columns=["scenario", "active_channels", "threshold", "AUC", "F1",
                                         "Precision", "Recall", "Specificity", "Accuracy", "errors"])
abl_df.to_csv(os.path.join(ABLATION_DIR, "ablation_summary.csv"), index=False)

summary = {"hyperparameters": {"dropout": 0.2, "lr": 3e-5, "weight_decay": 1e-4},
           "best_alpha": best_alpha,
           "alpha_sweep": sweep_df.to_dict(orient="records"),
           "best_alpha_test": sweep[best_alpha]["eval"]["test"],
           "ablation": abl_df.to_dict(orient="records")}
json.dump(summary, open(os.path.join(SWEEP_DIR, "alpha_ablation_summary.json"), "w"), indent=2)
print(f"\nSaved: {os.path.join(ABLATION_DIR, 'ablation_summary.csv')}")
print(f"Saved: {os.path.join(SWEEP_DIR, 'alpha_ablation_summary.json')}")

In [ ]:
# --- Fig 2.1 candidate explorer: largest-tumor patients per class ---
import numpy as np, matplotlib.pyplot as plt
S = CFG["img_size"]; N = 6   # how many candidates to show per class

def show_candidates(label, n=N):
    sub = (manifest[(manifest["rank"] == 0) & (manifest["idh_label"] == label)]
           .sort_values("tumor_area", ascending=False).head(n))
    fig, axes = plt.subplots(n, 3, figsize=(6.5, 2.1*n)); fig.patch.set_facecolor("white")
    axes = np.atleast_2d(axes)
    for r, (_, row) in enumerate(sub.iterrows()):
        d = np.load(row["slice_path"])
        for c, name in enumerate(["T2", "FLAIR", "ASL"]):
            axes[r, c].imshow(np.rot90(resize_slice(d[name].astype(np.float32), S, 1)),
                              cmap="gray", vmin=0, vmax=1)
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
            if r == 0: axes[r, c].set_title({"T2":"T2w","FLAIR":"FLAIR","ASL":"ASL"}[name], fontweight="bold")
        axes[r, 0].set_ylabel(f"{row['patient_id']}\narea={int(row['tumor_area'])}",
                              rotation=0, ha="right", va="center", fontsize=8)
    fig.suptitle(f"{'IDH-mutant' if label==1 else 'IDH-wildtype'} candidates (largest-tumor slice)",
                 fontweight="bold")
    plt.tight_layout(); plt.show()

show_candidates(0)   # wildtype
show_candidates(1)   # mutant

In [ ]:
# --- Fig 2.1 FINAL: 2 patients x (T2w/FLAIR/ASL), auto WT/Mutant ordering ---
PIDS = ["UCSF-PDGM-0065", "UCSF-PDGM-0501"]   # any order; one WT + one mutant

def pick(pid, k=None):
    rows = manifest[manifest["patient_id"] == pid]
    return (rows.sort_values("tumor_area", ascending=False).iloc[0] if k is None
            else rows.iloc[(rows["k_axial"] - k).abs().argmin()])

sel = sorted([pick(p) for p in PIDS], key=lambda r: int(r["idh_label"]))  # WT(0) top, Mut(1) bottom
S = CFG["img_size"]
fig, axes = plt.subplots(2, 3, figsize=(9.6, 6.6)); fig.patch.set_facecolor("white")
for r, row in enumerate(sel):
    d = np.load(row["slice_path"])
    rlabel = "IDH-mutant" if int(row["idh_label"]) == 1 else "IDH-wildtype"
    for c, name in enumerate(["T2", "FLAIR", "ASL"]):
        ax = axes[r, c]
        ax.imshow(np.rot90(resize_slice(d[name].astype(np.float32), S, 1)), cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        if r == 0: ax.set_title({"T2":"T2w","FLAIR":"FLAIR","ASL":"ASL"}[name], fontsize=15, fontweight="bold")
        if c == 0: ax.set_ylabel(rlabel, fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig2_1_sequences.pdf", bbox_inches="tight")            # vector labels - crispest in LaTeX
plt.savefig("fig2_1_sequences.png", dpi=300, bbox_inches="tight")   # high-res fallback
plt.show()
print("order:", [("Mut" if int(r["idh_label"])==1 else "WT") + " " + r["patient_id"] for r in sel])

In [ ]:
# --- Fig 4.2 setup: load BOTH trained models, build a comparator ---
import torch, numpy as np, matplotlib.pyplot as plt, os, json
NOMASK_CKPT = "/media/omeryasincur/Elements/omeryasin/pipeline_nomask3ch_allslices_results/best_model.pt"
SOFT_CKPT   = "/media/omeryasincur/Elements/omeryasin/pipeline_softmask3ch_grid/dp0.2_lr3.0e-05_wd1.0e-04/best_model.pt"
ALPHA = 0.5; S = CFG["img_size"]

def _load(ckpt):
    m = build_model(pretrained=False, dropout=0.2).to(DEVICE)
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE)); m.eval(); return m
model_nomask, model_soft = _load(NOMASK_CKPT), _load(SOFT_CKPT)
cam_nomask = GradCAMpp(model_nomask, model_nomask.layer4)   # create ONCE (hooks persist)
cam_soft   = GradCAMpp(model_soft,   model_soft.layer4)

def cams_for(pid, k=None):
    rows = manifest[manifest["patient_id"] == pid]
    row = (rows.sort_values("tumor_area", ascending=False).iloc[0] if k is None
           else rows.iloc[(rows["k_axial"]-k).abs().argmin()])
    d = np.load(row["slice_path"]); label = int(row["idh_label"])
    tumor = d["mask"].astype(np.float32); soft = ALPHA + (1.0-ALPHA)*tumor
    brain = resize_slice(d["brain"].astype(np.float32), S, 0)
    xp = torch.from_numpy(np.stack([resize_slice(d[m].astype(np.float32),      S, 1) for m in ["T2","FLAIR","ASL"]],0)).unsqueeze(0).to(DEVICE)
    xs = torch.from_numpy(np.stack([resize_slice(d[m].astype(np.float32)*soft, S, 1) for m in ["T2","FLAIR","ASL"]],0)).unsqueeze(0).to(DEVICE)
    hn, pn = cam_nomask(xp, target_class=label); hs, ps = cam_soft(xs, target_class=label)
    return (label, resize_slice(d["T2"].astype(np.float32), S, 1), resize_slice(tumor, S, 0),
            hn[0]*brain, hs[0]*brain, float(pn[0]), float(ps[0]), row["patient_id"], int(row["k_axial"]))

def compare(pid, k=None):
    label, t2, msk, hn, hs, pn, ps, pid_, kk = cams_for(pid, k)
    fig, ax = plt.subplots(1, 3, figsize=(10, 3.6)); fig.patch.set_facecolor("white")
    ax[0].imshow(np.rot90(t2), cmap="gray", vmin=0, vmax=1); ax[0].set_title("T2w + tumor")
    ax[0].contour(np.rot90(msk), [0.5], colors="lime", linewidths=1.3)
    for a,(hm,ti,p) in zip(ax[1:], [(hn,"no-mask",pn),(hs,"soft-mask (α=0.5)",ps)]):
        a.imshow(np.rot90(t2), cmap="gray", vmin=0, vmax=1); a.imshow(np.rot90(hm), cmap="jet", alpha=0.5)
        a.contour(np.rot90(msk), [0.5], colors="white", linewidths=1.3); a.set_title(f"{ti}  (p={p:.2f})")
    for a in ax: a.axis("off")
    fig.suptitle(f"{pid_} | {'Mut' if label==1 else 'WT'} | k={kk}", fontweight="bold")
    plt.tight_layout(); plt.show()

# candidates = test patients (falls back to split.json if test_mf isn't in memory)
try:    cand = list(dict.fromkeys(test_mf[test_mf["rank"]==0]["patient_id"]))
except NameError:
    cand = [p for p in json.load(open(os.path.join(SPLITS_DIR,"split.json")))["test"] if p in set(manifest["patient_id"])]
for pid in cand[:12]:   # scroll these: keep ones where soft-mask hugs the tumor & no-mask is scattered
    compare(pid)

In [ ]:
# --- Grad-CAM GALLERY: scan MANY patients at once (no-mask | soft-mask) ---
# Run AFTER the setup cell. Note the IDs you like, then put them in EXAMPLES (final cell).
import os, json
WHICH   = "all"   # "all" | "mut" | "wt"   -> filter by class
COUNT   = 48      # how many patients to show in this batch (bump it up for more)
START   = 0       # paging: 0, then 48, then 96, ... to walk through everyone
PER_ROW = 4       # patients per row (each patient = 2 mini-panels)

try:    pool = test_mf[test_mf["rank"] == 0].copy()
except NameError:
    tl = json.load(open(os.path.join(SPLITS_DIR, "split.json")))["test"]
    pool = manifest[(manifest["rank"] == 0) & (manifest["patient_id"].isin(tl))].copy()
if   WHICH == "mut": pool = pool[pool["idh_label"] == 1]
elif WHICH == "wt":  pool = pool[pool["idh_label"] == 0]
pids = pool.sort_values(["idh_label", "tumor_area"], ascending=[False, False])["patient_id"].tolist()
pids = pids[START:START + COUNT]
print(f"pool ({WHICH}) = {len(pool)} patients; showing {len(pids)} (from {START})")

ncols = PER_ROW * 2
nrows = int(np.ceil(len(pids) / PER_ROW))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2.0, nrows * 2.4)); fig.patch.set_facecolor("white")
axes = np.atleast_2d(axes)
for ax in axes.reshape(-1): ax.axis("off")
for i, pid in enumerate(pids):
    label, t2, msk, hn, hs, pn, ps, pid_, kk = cams_for(pid)
    r = i // PER_ROW; base = (i % PER_ROW) * 2
    cls  = "Mut" if label == 1 else "WT"
    tcol = "#1C7293" if label == 1 else "#B26A00"
    for j, (hm, tag, p) in enumerate([(hn, "no-mask", pn), (hs, "soft", ps)]):
        ax = axes[r, base + j]
        ax.imshow(np.rot90(t2), cmap="gray", vmin=0, vmax=1)
        ax.imshow(np.rot90(hm), cmap="jet", alpha=0.5)
        ax.contour(np.rot90(msk), [0.5], colors="white", linewidths=1.0)
        ax.set_title(f"{pid_} - {cls}\n{tag} p={p:.2f}" if j == 0 else f"{tag} p={p:.2f}",
                     fontsize=8, fontweight="bold", color=(tcol if j == 0 else "black"))
plt.tight_layout(); plt.show()

In [ ]:
# --- Fig 4.2 FINAL: chosen examples, no-mask vs soft-mask Grad-CAM (no p-values) ---
EXAMPLES = ["UCSF-PDGM-0021", "UCSF-PDGM-0324", "UCSF-PDGM-0305",
            "UCSF-PDGM-0510", "UCSF-PDGM-0033", "UCSF-PDGM-0487"]

n = len(EXAMPLES)
fig, axes = plt.subplots(n, 2, figsize=(5.6, 2.6 * n)); fig.patch.set_facecolor("white")
axes = np.atleast_2d(axes)
titles = ["Grad-CAM (no-mask)", r"Grad-CAM (soft-mask, $\alpha$ = 0.5)"]
for r, pid in enumerate(EXAMPLES):
    label, t2, msk, hn, hs, pn, ps, pid_, kk = cams_for(pid)   # p-values (pn, ps) ignored
    rlabel = f"{pid_}\n{'IDH-mutant' if label == 1 else 'IDH-wildtype'}"
    for c, hm in enumerate([hn, hs]):
        ax = axes[r, c]
        ax.imshow(np.rot90(t2), cmap="gray", vmin=0, vmax=1)
        ax.imshow(np.rot90(hm), cmap="jet", alpha=0.5)
        ax.contour(np.rot90(msk), [0.5], colors="white", linewidths=1.2)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        if r == 0: ax.set_title(titles[c], fontsize=12, fontweight="bold")
        if c == 0: ax.text(-0.06, 0.5, rlabel, transform=ax.transAxes,
                           ha="right", va="center", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig("fig4_2_gradcam.pdf", bbox_inches="tight")            # vector labels - best for LaTeX
plt.savefig("fig4_2_gradcam.png", dpi=300, bbox_inches="tight")   # high-res fallback
plt.show()
print("saved fig4_2_gradcam.pdf / .png")

In [ ]:
# --- Fig 4.2 FINAL (v3): only the patient ID as the row label (no class / no p-values) ---
EXAMPLES = ["UCSF-PDGM-0021", "UCSF-PDGM-0324", "UCSF-PDGM-0305",
            "UCSF-PDGM-0510", "UCSF-PDGM-0033", "UCSF-PDGM-0487"]

n = len(EXAMPLES)
fig, axes = plt.subplots(n, 2, figsize=(6.8, 2.7 * n)); fig.patch.set_facecolor("white")
axes = np.atleast_2d(axes)
titles = ["No-mask", r"Soft-mask ($\alpha$ = 0.5)"]
for r, pid in enumerate(EXAMPLES):
    _, t2, msk, hn, hs, pn, ps, pid_, kk = cams_for(pid)
    for c, hm in enumerate([hn, hs]):
        ax = axes[r, c]
        ax.imshow(np.rot90(t2), cmap="gray", vmin=0, vmax=1)
        ax.imshow(np.rot90(hm), cmap="jet", alpha=0.5)
        ax.contour(np.rot90(msk), [0.5], colors="white", linewidths=1.2)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        if r == 0: ax.set_title(titles[c], fontsize=13, fontweight="bold")
        if c == 0: ax.text(-0.07, 0.5, pid_, transform=ax.transAxes, ha="right", va="center",
                           fontsize=11, fontweight="bold")     # <-- only the dataset ID
plt.tight_layout()
plt.savefig("fig4_2_gradcam.pdf", bbox_inches="tight")
plt.savefig("fig4_2_gradcam.png", dpi=300, bbox_inches="tight")
plt.show(); print("saved fig4_2_gradcam.pdf / .png")

In [ ]:
# --- Fig 4.2 FINAL (v4): punchier heatmaps (higher opacity + mild contrast) ---
EXAMPLES = ["UCSF-PDGM-0021", "UCSF-PDGM-0324", "UCSF-PDGM-0305",
            "UCSF-PDGM-0510", "UCSF-PDGM-0033", "UCSF-PDGM-0487"]
ALPHA_OVERLAY = 0.65   # heatmap opacity (0.5 looked washed out; try 0.55-0.75)
GAMMA         = 0.85   # <1 pushes attention toward warm colors (more pop); 1.0 = off

n = len(EXAMPLES)
fig, axes = plt.subplots(n, 2, figsize=(6.8, 2.7 * n)); fig.patch.set_facecolor("white")
axes = np.atleast_2d(axes)
titles = ["No-mask", r"Soft-mask ($\alpha$ = 0.5)"]
for r, pid in enumerate(EXAMPLES):
    _, t2, msk, hn, hs, pn, ps, pid_, kk = cams_for(pid)
    for c, hm in enumerate([hn, hs]):
        ax = axes[r, c]
        h = np.rot90(np.clip(hm, 0, 1) ** GAMMA)
        ax.imshow(np.rot90(t2), cmap="gray", vmin=0, vmax=1)
        ax.imshow(h, cmap="jet", alpha=ALPHA_OVERLAY, vmin=0, vmax=1)
        ax.contour(np.rot90(msk), [0.5], colors="white", linewidths=1.2)
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        if r == 0: ax.set_title(titles[c], fontsize=13, fontweight="bold")
        if c == 0: ax.text(-0.07, 0.5, pid_, transform=ax.transAxes, ha="right", va="center",
                           fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig("fig4_2_gradcam.pdf", bbox_inches="tight")
plt.savefig("fig4_2_gradcam.png", dpi=300, bbox_inches="tight")
plt.show(); print("saved fig4_2_gradcam.pdf / .png")

In [ ]:
# --- Appendix B figure: training curves (primary soft-mask alpha=0.5 model) ---
import json, matplotlib.pyplot as plt
HIST = "/media/omeryasincur/Elements/omeryasin/pipeline_softmask3ch_grid/dp0.2_lr3.0e-05_wd1.0e-04/history.json"
h = json.load(open(HIST)); print("keys:", list(h.keys()))   # expect: epoch, train_loss, val_loss, val_auc
ep = h["epoch"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8)); fig.patch.set_facecolor("white")
ax[0].plot(ep, h["train_loss"], lw=2, label="Train")
ax[0].plot(ep, h["val_loss"],   lw=2, label="Validation")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss")
ax[0].set_title("Training and validation loss", fontweight="bold"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(ep, h["val_auc"], lw=2, color="#1C7293")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("AUC")
ax[1].set_title("Validation AUC", fontweight="bold"); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("fig_training_curves.pdf", bbox_inches="tight")
plt.savefig("fig_training_curves.png", dpi=300, bbox_inches="tight")
plt.show(); print("saved fig_training_curves.pdf / .png")

In [ ]:
# --- Soft-mask illustration: same slice at different alpha ---
import numpy as np, matplotlib.pyplot as plt
PID    = "UCSF-PDGM-0021"     # net tümörlü bir vaka (istersen 0065 ya da 0501 - Fig 2.1'dekiler)
MOD    = "T2"                 # cache anahtarı: "T2" / "FLAIR" / "ASL"
ALPHAS = [0.3, 0.5, 0.7]

row  = manifest[(manifest.patient_id == PID) & (manifest["rank"] == 0)].iloc[0]
d    = np.load(row["slice_path"])
img  = d[MOD].astype("float32")
mask = d["mask"].astype("float32")

panels = [("No mask", img)]
for a in ALPHAS:
    m = a + (1.0 - a) * mask          # tumor -> 1, background -> alpha
    panels.append((rf"$\alpha = {a}$", img * m))

fig, ax = plt.subplots(1, len(panels), figsize=(3.0 * len(panels), 3.2)); fig.patch.set_facecolor("white")
for k, (title, im) in enumerate(panels):
    ax[k].imshow(np.rot90(im), cmap="gray", vmin=0, vmax=1)
    ax[k].contour(np.rot90(mask), [0.5], colors="#FFD400", linewidths=1.3)   # tumor outline
    ax[k].set_title(title, fontsize=13, fontweight="bold")
    ax[k].set_xticks([]); ax[k].set_yticks([])
    for s in ax[k].spines.values(): s.set_visible(False)
plt.tight_layout()
plt.savefig("fig_softmask_alpha.pdf", bbox_inches="tight")
plt.savefig("fig_softmask_alpha.png", dpi=300, bbox_inches="tight")
plt.show(); print("saved fig_softmask_alpha.pdf / .png")